### Proyek UAS IBDA 2112 - Keamanan dan Pengelolaan Data

**Pemrograman No 2 - Anonimisasi (differential privacy)**

Kelompok:
- Alfandi Wijaya - 232200156
- Keanrich Cordana - 232301226

Komitmen Integritas

“Di hadapan TUHAN yang hidup, saya menegaskan bahwa saya tidak memberikan maupun menerima bantuan apapun—baik lisan, tulisan, maupun elektronik—di dalam ujian ini selain daripada apa yang telah diizinkan oleh pengajar, dan tidak akan menyebarkan baik soal maupun jawaban ujian kepada pihak lain.”

In [1]:
! python --version

Python 3.11.12


In [2]:
!apt install python3.11 python3-pip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3.11 is already the newest version (3.11.12-1+jammy1).
The following additional packages will be installed:
  python3-setuptools python3-wheel
Suggested packages:
  python-setuptools-doc
The following NEW packages will be installed:
  python3-pip python3-setuptools python3-wheel
0 upgraded, 3 newly installed, 0 to remove and 34 not upgraded.
Need to get 1,677 kB of archives.
After this operation, 8,968 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-setuptools all 59.6.0-1.2ubuntu0.22.04.2 [340 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 python3-wheel all 0.37.1-2ubuntu0.22.04.1 [32.0 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 python3-pip all 22.0.2+dfsg-1ubuntu0.5 [1,306 kB]
Fetched 1,677 kB in 2s (820 kB/s)
Selecting previously unselected package python3-setupto

In [3]:
!pip install tmlt.analytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.9/161.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 931.1/931.1 kB 38.3 MB/s eta 0:00:00
  Attempting uninstall: tabulate
    Found existing installation: tabulate 0.9.0
    Uninstalling tabulate-0.9.0:
      Successfully uninstalled tabulate-0.9.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.1
    Uninstalling sympy-1.13.1:
      Successfully u

In [66]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pyspark.sql import SparkSession
from tmlt.analytics.keyset import KeySet
from tmlt.analytics.privacy_budget import PureDPBudget
from tmlt.analytics.protected_change import AddOneRow
from tmlt.analytics.query_builder import QueryBuilder, ColumnType, BinningSpec
from tmlt.analytics.session import Session


spark = SparkSession.builder.getOrCreate()
data_df = spark.read.csv("safe_data.csv", header=True, inferSchema=True)

In [67]:
data_df.head(5)

[Row(Id='32c84703-2481-49cd-d571-3899d5820253', BIRTHDATE='46-ZyNbka9a8w9y91dmisQ==', PREFIX='M/F', FIRST='6rLFyJesgtTPr77H1NmisQ==', LAST='6tTc55nYb9y6rqui', MAIDEN='p8jFxA==', MARITAL='M', GENDER='M/F', ADDRESS='5K_Mx9DmaM_Rx9K3luK041HQrdKomdHM37VdqA==', CITY='Boston', START='datetime', STOP='satetime', ENCOUNTERCLASS='CLASS', DESCRIPTION='DESCRIPTION', BASE_ENCOUNTER_COST=85.55, TOTAL_CLAIM_COST=1018.02, PAYER_COVERAGE=0.0, REASONDESCRIPTION='ILLNESS'),
 Row(Id='c98059da-320a-c0a6-fced-c8815f3e3f39', BIRTHDATE='46-ZzNXka9a609y71dmisQ==', PREFIX='M/F', FIRST='6rOayJrnV9m76ejJ', LAST='6Z2axJbnmtjN3eC91ePMsQ==', MAIDEN='57OzupathtnPsuC81ePIsQ==', MARITAL='M', GENDER='M/F', ADDRESS='5JrYxtDlmtXPxtK5lue8upSYucCmvt3DlutdqA==', CITY='Boston', START='datetime', STOP='satetime', ENCOUNTERCLASS='CLASS', DESCRIPTION='DESCRIPTION', BASE_ENCOUNTER_COST=142.58, TOTAL_CLAIM_COST=2619.36, PAYER_COVERAGE=0.0, REASONDESCRIPTION='ILLNESS'),
 Row(Id='4ad28a3a-2479-782b-f29c-d5b3f41a001e', BIRTHDATE='46

In [68]:
data=data_df.toPandas()
data.head()

,Id,BIRTHDATE,PREFIX,FIRST,LAST,MAIDEN,MARITAL,GENDER,ADDRESS,CITY,START,STOP,ENCOUNTERCLASS,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONDESCRIPTION
0,32c84703-2481-49cd-d571-3899d5820253,46-ZyNbka9a8w9y91dmisQ==,M/F,6rLFyJesgtTPr77H1NmisQ==,6tTc55nYb9y6rqui,p8jFxA==,M,M/F,5K_Mx9DmaM_Rx9K3luK041HQrdKomdHM37VdqA==,Boston,datetime,satetime,CLASS,DESCRIPTION,85.55,1018.02,0.00,ILLNESS
1,c98059da-320a-c0a6-fced-c8815f3e3f39,46-ZzNXka9a609y71dmisQ==,M/F,6rOayJrnV9m76ejJ,6Z2axJbnmtjN3eC91ePMsQ==,57OzupathtnPsuC81ePIsQ==,M,M/F,5JrYxtDlmtXPxtK5lue8upSYucCmvt3DlutdqA==,Boston,datetime,satetime,CLASS,DESCRIPTION,142.58,2619.36,0.00,ILLNESS
2,4ad28a3a-2479-782b-f29c-d5b3f41a001e,46-ZyNXka9a76dy-1MOisQ==,M/F,6rLFyJesgtTP7ba-1KjUsQ==,6siau5nCdN276ra9,7NiWv5rnkde806ei,M,M/F,5NXYxtDmZNLN3ce50NW91Zm1sOaor7a-0sBlqA==,Boston,datetime,satetime,CLASS,DESCRIPTION,85.55,461.59,305.27,ILLNESS
3,c3f4da61-e4b4-21d5-587a-fbc89943bc19,46-ZyNXOa9a709y91NmisQ==,M/F,6LLZyJWsV9S6ruzI,552auJbBktjRxMbK1MOisQ==,p8jFxA==,M,M/F,45rcy9DmmtvP7aqul8y0pFLPy9GorqvXl7VdqA==,Boston,datetime,satetime,CLASS,DESCRIPTION,136.80,1784.24,0.00,ILLNESS
4,a9183b4f-2572-72ea-54c2-b3cd038b4be7,46-Zy9W-a9e609y71OmisQ==,M/F,6LOvu5rCitXP6sLH1tmisQ==,6dnJu5esmsXQ6aDG1rOisQ==,p8jFxA==,M,M/F,5K-VzNDmjtXQ7emzm-K04lK6ucY=,Braintree,datetime,satetime,CLASS,DESCRIPTION,85.55,234.72,0.00,ILLNESS


In [69]:
session = Session.from_dataframe(
    privacy_budget=PureDPBudget(epsilon=1.1),
    source_id="data_safe",
    dataframe=data_df,
    protected_change=AddOneRow(),
)
data_df.columns

['Id',
 'BIRTHDATE',
 'PREFIX',
 'FIRST',
 'LAST',
 'MAIDEN',
 'MARITAL',
 'GENDER',
 'ADDRESS',
 'CITY',
 'START',
 'STOP',
 'ENCOUNTERCLASS',
 'DESCRIPTION',
 'BASE_ENCOUNTER_COST',
 'TOTAL_CLAIM_COST',
 'PAYER_COVERAGE',
 'REASONDESCRIPTION']

In [70]:
city_list = data_df.select("CITY").distinct().rdd.flatMap(lambda x: x).collect()
print(city_list)

city_levels = KeySet.from_dict({
    "CITY": city_list      # ganti "CITY" sesuai nama kolom di data_safe
})


['Everett', 'Hull', 'Norwell', 'Cambridge', 'Medford', 'Stoneham', 'Newton', 'Winchester', 'Milton', 'Weymouth', 'Lynnfield', 'Winthrop', 'North Scituate', 'Belmont', 'Cohasset', 'Reading', 'Melrose', 'Braintree', 'Quincy', 'Somerville', 'Revere', 'Hingham', 'Chelsea', 'Brookline', 'Watertown', 'Scituate', 'Malden', 'Boston', 'Waltham']


In [71]:
average_Base_Cost_query = (
    QueryBuilder("data_safe")
    .groupby(city_levels)
    .average("BASE_ENCOUNTER_COST", low=0, high=10000)
)
encounter_average_cost = session.evaluate(
    average_Base_Cost_query,
    privacy_budget=PureDPBudget(0.6),
)
encounter_average_cost.sort("BASE_ENCOUNTER_COST_average", ascending=False).show(truncate=False)

/usr/local/lib/python3.11/dist-packages/pyspark/sql/pandas/functions.py:407: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(


+--------------+---------------------------+
|CITY          |BASE_ENCOUNTER_COST_average|
+--------------+---------------------------+
|Belmont       |4572.179275380166          |
|Milton        |4477.458365411154          |
|Winchester    |1749.4458948509432         |
|Lynnfield     |1268.8852266548602         |
|North Scituate|711.0207555020961          |
|Brookline     |457.56426242145517         |
|Somerville    |278.33819939074056         |
|Braintree     |252.0894112900287          |
|Hingham       |219.22995417095444         |
|Reading       |206.46502453483117         |
|Revere        |185.6930211419267          |
|Everett       |143.89413079550923         |
|Cambridge     |134.01368567640202         |
|Scituate      |131.9671853693162          |
|Winthrop      |124.92041320321732         |
|Chelsea       |124.64508528920942         |
|Boston        |116.09030324836112         |
|Weymouth      |99.52161921601055          |
|Hull          |83.11051613756081          |
|Quincy   